In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")
# Updated file paths to use the 'dataset' folder
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

def advanced_feature_engineering(df):
    """Creates powerful new features for the model to learn from"""
    df = df.copy()
    
    # 1. Base Time Features
    df['hour'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else -1)
    df['minute'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]) if pd.notnull(x) else -1)
    df['time_in_mins'] = df['hour'] * 60 + df['minute']
    
    # 2. Cyclical Time (so the model knows 23:59 and 00:01 are close together)
    df['sin_time'] = np.sin(2 * np.pi * df['time_in_mins'] / 1440)
    df['cos_time'] = np.cos(2 * np.pi * df['time_in_mins'] / 1440)
    
    df = df.drop(['timestamp'], axis=1)
    return df

print("Applying feature engineering...")
train_fe = advanced_feature_engineering(train)
test_fe = advanced_feature_engineering(test)

# 3. Target Encoding for Geohash (Highly Effective for locations)
print("Applying Target Encoding to Geohash...")
global_mean = train_fe['demand'].mean()
geohash_stats = train_fe.groupby('geohash')['demand'].agg(['mean', 'count'])

# Smoothing factor prevents overfitting on locations with very few data points
weight = 20 
geohash_stats['smoothed_demand'] = (geohash_stats['count'] * geohash_stats['mean'] + weight * global_mean) / (geoh

Loading datasets...
Preprocessing data...
Training LightGBM model (this might take a moment)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001137 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 560
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 10
[LightGBM] [Info] Start training from score 0.093942
Generating predictions...
Success! Predictions saved to 'dataset/submission.csv'. You can now download this file and submit it to HackerEarth.
